In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("sparkPartitionsApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/15 15:18:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
sc.defaultParallelism

4

In [6]:
# Read taxi zones data
taxiZonesDf = spark.read.option("inferSchema", "true").csv(
    "./Files/TaxiZones.csv"
)

# Read the number of partitions
print("Partitions = " + str(taxiZonesDf.rdd.getNumPartitions()))

# Check the number of records
print("Record count = " + str(taxiZonesDf.count()))

Partitions = 1
Record count = 265


In [23]:
# Read yellow taxis data
yellowTaxisDf = spark.read.option("inferSchema", "true").csv(
    "./Files/YellowTaxis_202210.csv",
    header=True
)

# Deafult parallelism
print("Default parallelism = " + str(sc.defaultParallelism))

# Read the number of partitions
print("Partitions = " + str(yellowTaxisDf.rdd.getNumPartitions()))

# Check the number of records
print("Record count = " + str(yellowTaxisDf.count()))

yellowTaxisDf.printSchema()

Default parallelism = 4
Partitions = 7
Record count = 3675412
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [8]:
spark.conf.set("spark.sql.files.maxPartitionBytes", "64m")

In [21]:
def getDataFrameStats(dataFrame, columnName):
    outputDf = (
                    dataFrame
                        # Get partition number for each record
                        .withColumn("Partition Number", spark_partition_id())
                        # Group by patitioon and calculate stats for a column
                        .groupBy("Partition Number")
                        .agg(
                            count("*").alias("Record Count"),
                            min(columnName).alias("Min Column Value " + columnName),
                            max(columnName).alias("Max Column Value " + columnName),
                        )
                        .orderBy("Partition Number")
    )
    return outputDf

In [24]:
getDataFrameStats(yellowTaxisDf, "PULocationID").show()

+----------------+------------+-----------------------------+-----------------------------+
|Partition Number|Record Count|Min Column Value PULocationID|Max Column Value PULocationID|
+----------------+------------+-----------------------------+-----------------------------+
|               0|      531991|                            1|                          265|
|               1|      531721|                            1|                          265|
|               2|      531579|                            1|                          265|
|               3|      531536|                            1|                          265|
|               4|      531728|                            1|                          265|
|               5|      531528|                            1|                          265|
|               6|      485329|                            1|                          265|
+----------------+------------+-----------------------------+-------------------

In [25]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

In [26]:
# Group the data
yellowTaxisGroupedDf = (
    yellowTaxisDf
        .groupBy("PULocationID")
        .agg(sum("total_amount"))
)

# Check the number of partitions
print("Partitions after group by = " + str(yellowTaxisGroupedDf.rdd.getNumPartitions()))

# Get dataframe stats
getDataFrameStats(yellowTaxisGroupedDf, "PULocationID").show()

Partitions after group by = 200


+----------------+------------+-----------------------------+-----------------------------+
|Partition Number|Record Count|Min Column Value PULocationID|Max Column Value PULocationID|
+----------------+------------+-----------------------------+-----------------------------+
|               0|           1|                          148|                          148|
|               1|           1|                          243|                          243|
|               2|           1|                           31|                           31|
|               3|           3|                           85|                          251|
|               4|           1|                           65|                           65|
|               5|           2|                           53|                          255|
|               6|           1|                          133|                          133|
|               7|           1|                           78|                   

In [27]:
spark.conf.set("spark.sql.shuffle.partitions", 3)

In [28]:
# Group the data
yellowTaxisGroupedDf = (
    yellowTaxisDf
        .groupBy("PULocationID")
        .agg(sum("total_amount"))
)

# Check the number of partitions
print("Partitions after group by = " + str(yellowTaxisGroupedDf.rdd.getNumPartitions()))

# Get dataframe stats
getDataFrameStats(yellowTaxisGroupedDf, "PULocationID").show()

Partitions after group by = 3


+----------------+------------+-----------------------------+-----------------------------+
|Partition Number|Record Count|Min Column Value PULocationID|Max Column Value PULocationID|
+----------------+------------+-----------------------------+-----------------------------+
|               0|          88|                            3|                          263|
|               1|          85|                            1|                          265|
|               2|          87|                           11|                          264|
+----------------+------------+-----------------------------+-----------------------------+



In [29]:
getDataFrameStats(yellowTaxisDf, "PULocationID").show()

+----------------+------------+-----------------------------+-----------------------------+
|Partition Number|Record Count|Min Column Value PULocationID|Max Column Value PULocationID|
+----------------+------------+-----------------------------+-----------------------------+
|               0|      531991|                            1|                          265|
|               1|      531721|                            1|                          265|
|               2|      531579|                            1|                          265|
|               3|      531536|                            1|                          265|
|               4|      531728|                            1|                          265|
|               5|      531528|                            1|                          265|
|               6|      485329|                            1|                          265|
+----------------+------------+-----------------------------+-------------------

In [ ]:
# Round robin partitioning
roundRobinRepartitionedDF1 = yellowTaxisDf.repartition(14)
getDataFrameStats(roundRobinRepartitionedDF1, "PULocationID").show()

+----------------+------------+-----------------------------+-----------------------------+
|Partition Number|Record Count|Min Column Value PULocationID|Max Column Value PULocationID|
+----------------+------------+-----------------------------+-----------------------------+
|               0|      262529|                            1|                          265|
|               1|      262529|                            1|                          265|
|               2|      262530|                            1|                          265|
|               3|      262531|                            1|                          265|
|               4|      262531|                            1|                          265|
|               5|      262531|                            1|                          265|
|               6|      262531|                            1|                          265|
|               7|      262530|                            1|                   

In [31]:
# Hash partitioning
hashPartitionedDF = yellowTaxisDf.repartition("PULocationID")
getDataFrameStats(hashPartitionedDF, "PULocationID").show()

+----------------+------------+-----------------------------+-----------------------------+
|Partition Number|Record Count|Min Column Value PULocationID|Max Column Value PULocationID|
+----------------+------------+-----------------------------+-----------------------------+
|               0|     1246579|                            3|                          263|
|               1|     1426282|                            1|                          265|
|               2|     1002551|                           11|                          264|
+----------------+------------+-----------------------------+-----------------------------+



In [32]:
# Range partitioning
rangePartitionedDF = yellowTaxisDf.repartitionByRange("PULocationID")
getDataFrameStats(rangePartitionedDF, "PULocationID").show()

+----------------+------------+-----------------------------+-----------------------------+
|Partition Number|Record Count|Min Column Value PULocationID|Max Column Value PULocationID|
+----------------+------------+-----------------------------+-----------------------------+
|               0|     1272694|                            1|                          140|
|               1|     1340481|                          141|                          230|
|               2|     1062237|                          231|                          265|
+----------------+------------+-----------------------------+-----------------------------+



In [33]:
coalescedDF = yellowTaxisDf.coalesce(2)
getDataFrameStats(coalescedDF, "PULocationID").show()

+----------------+------------+-----------------------------+-----------------------------+
|Partition Number|Record Count|Min Column Value PULocationID|Max Column Value PULocationID|
+----------------+------------+-----------------------------+-----------------------------+
|               0|     1595291|                            1|                          265|
|               1|     2080121|                            1|                          265|
+----------------+------------+-----------------------------+-----------------------------+



25/06/15 20:08:05 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 890616 ms exceeds timeout 120000 ms
25/06/15 20:08:05 WARN SparkContext: Killing executors is not supported by current scheduler.
25/06/15 20:08:06 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$